# 06 — Bias Correction and Multi-Model Ensemble Analysis
## Part 5 — Uncertainty, Weighting, Trends, Extremes, and Production Workflow

This final part adds uncertainty analysis, model weighting, seasonal and extreme-event diagnostics, trend uncertainty, agreement analysis, publication figures, and a reproducible production workflow.

## Objectives

1. Evaluate model performance using independent validation data.
2. Build equal-weight and skill-weighted ensembles.
3. Quantify model spread and uncertainty.
4. Compare historical and future climate-change signals.
5. Analyze seasonal and annual behavior.
6. Extend the workflow to climate extremes.
7. Estimate model trends and trend uncertainty.
8. Produce transparent, publication-ready diagnostics.
9. Save reproducible NetCDF and CSV outputs.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

In [ ]:
BASE = Path('.')
OUTPUT_DIR = BASE / 'outputs' / 'bias_correction'
CORRECTED_DIR = OUTPUT_DIR / 'corrected_models'
ENSEMBLE_DIR = OUTPUT_DIR / 'ensemble'
METRICS_DIR = OUTPUT_DIR / 'metrics'
FIGURE_DIR = OUTPUT_DIR / 'diagnostics'
VARIABLE = 'tasmax'
SCENARIO = 'rcp26'
HIST_START, HIST_END = '1981-01-01', '2000-12-31'
FUT_START, FUT_END = '2081-01-01', '2100-12-31'
print('Output directory:', OUTPUT_DIR)

## 1. Load validation metrics

Model weights should be derived from independent validation performance, not from future projection performance.

In [ ]:
metrics_file = METRICS_DIR / f'{VARIABLE}_{SCENARIO}_validation_metrics.csv'
if metrics_file.exists():
    validation = pd.read_csv(metrics_file)
    display(validation.head())
else:
    validation = pd.DataFrame()
    print('Validation file not found. Run Part 4 first.')

## 2. Build a model-skill table

In [ ]:
def make_skill_table(df):
    if df.empty: return pd.DataFrame()
    out = df.copy()
    rmse_col = next((c for c in ['corrected_rmse','rmse_corrected','RMSE_corrected','rmse'] if c in out.columns), None)
    if rmse_col is None: return out
    out['skill_rmse'] = pd.to_numeric(out[rmse_col], errors='coerce')
    return out

skill_table = make_skill_table(validation)
display(skill_table.head() if not skill_table.empty else skill_table)

## 3. Equal-weight ensemble

Equal weighting is the recommended baseline because it is transparent and does not impose assumptions about relative model skill.

In [ ]:
def equal_weights(models):
    models = list(models)
    return pd.Series(1 / len(models), index=models) if models else pd.Series(dtype=float)

if not skill_table.empty and 'model' in skill_table:
    models = skill_table['model'].dropna().unique()
    equal_w = equal_weights(models)
    display(equal_w.to_frame('weight'))

## 4. Skill-based weighting

A simple inverse-RMSE scheme is: wᵢ = (1/Eᵢ) / Σ(1/Eⱼ). Use this primarily as a sensitivity analysis.

In [ ]:
def inverse_error_weights(errors):
    errors = pd.Series(errors, dtype=float).replace([np.inf,-np.inf], np.nan).dropna().clip(lower=1e-12)
    raw = 1 / errors
    return raw / raw.sum()

if not skill_table.empty and 'model' in skill_table and 'skill_rmse' in skill_table:
    errors = skill_table.groupby('model')['skill_rmse'].mean()
    skill_weights = inverse_error_weights(errors)
    display(skill_weights.sort_values(ascending=False).to_frame('weight'))

## 5. Prevent excessive dominance by one model

In [ ]:
def cap_weights(weights, max_weight=0.5):
    w = pd.Series(weights, dtype=float).clip(lower=0)
    if w.sum() == 0: return w
    w = w / w.sum()
    for _ in range(20):
        excess = (w - max_weight).clip(lower=0).sum()
        w = w.clip(upper=max_weight)
        if excess <= 1e-12: break
        free = w < max_weight - 1e-12
        if not free.any(): break
        w.loc[free] += excess * w.loc[free] / w.loc[free].sum()
    return w / w.sum()

## 6. Load corrected model files

In [ ]:
def load_corrected_models(directory, variable):
    arrays = []
    for f in sorted(Path(directory).glob('*.nc')):
        ds = xr.open_dataset(f)
        if variable not in ds:
            print('Skipping:', f.name)
            continue
        da = ds[variable].load().expand_dims(model=[f.stem.replace('_corrected','')])
        arrays.append(da)
    if not arrays: raise FileNotFoundError(f'No corrected NetCDF files found in {directory}')
    return xr.concat(arrays, dim='model', join='exact', coords='minimal', compat='equals')

# corrected_models = load_corrected_models(CORRECTED_DIR, VARIABLE)

## 7. Weighted ensemble mean

In [ ]:
def weighted_ensemble_mean(data, weights):
    weights = pd.Series(weights, dtype=float)
    common = [m for m in data.model.values if m in weights.index]
    if not common: raise ValueError('No model names overlap between data and weights.')
    subset = data.sel(model=common)
    w = xr.DataArray(weights.loc[common].values, dims=['model'], coords={'model':common})
    return (subset * (w / w.sum())).sum('model')

# weighted_mean = weighted_ensemble_mean(corrected_models, skill_weights)

## 8. Compare equal and skill-weighted ensembles

In [ ]:
def summarize_weighting_difference(equal_mean, weighted_mean):
    d = weighted_mean - equal_mean
    return {'mean_difference': float(d.mean()), 'mean_absolute_difference': float(np.abs(d).mean()), 'maximum_absolute_difference': float(np.abs(d).max())}

# equal_mean = corrected_models.mean('model')
# weighted_mean = weighted_ensemble_mean(corrected_models, skill_weights)
# summarize_weighting_difference(equal_mean, weighted_mean)

## 9. Model spread

Standard deviation, interquartile range, and minimum–maximum range describe disagreement among models.

In [ ]:
def model_spread(data): return data.std('model')
def model_iqr(data): return data.quantile(0.75, dim='model') - data.quantile(0.25, dim='model')
def model_range(data): return data.max('model') - data.min('model')

## 10. Historical and future climate-change signal

In [ ]:
def climate_change_signal(data, hist_start, hist_end, fut_start, fut_end):
    hist = data.sel(time=slice(hist_start,hist_end)).mean('time')
    fut = data.sel(time=slice(fut_start,fut_end)).mean('time')
    return fut - hist

# change_by_model = climate_change_signal(corrected_models, HIST_START, HIST_END, FUT_START, FUT_END)

## 11. Model agreement on the sign of change

In [ ]:
def positive_agreement(data): return (data > 0).mean('model')
def negative_agreement(data): return (data < 0).mean('model')

def classify_agreement(agreement):
    return xr.where(agreement >= 0.80, 3, xr.where(agreement >= 0.60, 2, xr.where(agreement >= 0.50, 1, 0)))

## 12. Seasonal analysis

Annual statistics can hide important seasonal changes.

In [ ]:
def seasonal_mean(data): return data.groupby('time.season').mean('time')
def seasonal_change(data, hist_start, hist_end, fut_start, fut_end):
    h = data.sel(time=slice(hist_start,hist_end)).groupby('time.season').mean('time')
    f = data.sel(time=slice(fut_start,fut_end)).groupby('time.season').mean('time')
    return f - h

## 13. Annual diagnostics

In [ ]:
def annual_mean(data): return data.groupby('time.year').mean('time')
def annual_max(data): return data.groupby('time.year').max('time')
def annual_min(data): return data.groupby('time.year').min('time')

## 14. Annual model trends

Linear trend is a simple diagnostic. For publication, a robust estimator such as Sen's slope plus an appropriate significance test can be added.

In [ ]:
def linear_trend_1d(values, years):
    values, years = np.asarray(values,float), np.asarray(years,float)
    valid = np.isfinite(values) & np.isfinite(years)
    if valid.sum() < 3: return np.nan
    return np.polyfit(years[valid], values[valid], 1)[0]

def model_trends(annual_data):
    return xr.apply_ufunc(linear_trend_1d, annual_data, annual_data['year'], input_core_dims=[['year'],['year']], output_core_dims=[[]], vectorize=True, dask='parallelized', output_dtypes=[float])

## 15. Trend uncertainty

In [ ]:
def trend_summary(trends):
    return xr.Dataset({'trend_mean':trends.mean('model'),'trend_median':trends.median('model'),'trend_std':trends.std('model'),'trend_min':trends.min('model'),'trend_max':trends.max('model')})

def trend_agreement(trends):
    return xr.Dataset({'positive_fraction':(trends>0).mean('model'),'negative_fraction':(trends<0).mean('model')})

## 16. Climate-extreme indicators

The same ensemble framework can be applied to annual ETCCDI indicators such as TXx, TX90p, WSDI, RX1day, RX5day, R95p, R99p, CDD, and CWD.

In [ ]:
EXTREME_INDICATORS = ['TXx','TNx','TX90p','WSDI','RX1day','RX5day','R95p','R99p','CDD','CWD']
print('Indicators:', ', '.join(EXTREME_INDICATORS))

## 17. Ensemble statistics for indicators

In [ ]:
def indicator_ensemble_statistics(model_data):
    return xr.Dataset({'mean':model_data.mean('model'),'median':model_data.median('model'),'std':model_data.std('model'),'min':model_data.min('model'),'max':model_data.max('model'),'p10':model_data.quantile(.10,dim='model'),'p90':model_data.quantile(.90,dim='model')})

## 18. Threshold exceedance agreement

In [ ]:
def fraction_above(data, threshold): return (data > threshold).mean('model')
def fraction_below(data, threshold): return (data < threshold).mean('model')

## 19. Uncertainty decomposition framework

Internal variability, model uncertainty, and scenario uncertainty are conceptually distinct. The present workflow directly estimates model uncertainty from model spread. Scenario uncertainty requires multiple scenarios, while internal variability requires multiple realizations or suitable statistical analysis.

## 20. Scenario uncertainty

In [ ]:
def scenario_difference(scenario_a, scenario_b): return scenario_b - scenario_a
# scenario_signal = scenario_difference(ensemble_rcp26, ensemble_rcp85)

## 21. Effective model count

For weighted ensembles, N_eff = 1 / Σwᵢ². A low value indicates that a small number of models dominate the weighted result.

In [ ]:
def effective_model_count(weights):
    w = pd.Series(weights,dtype=float)
    w = w[w>0] / w[w>0].sum()
    return 1 / np.sum(w**2)

## 22. Distributional validation

For extremes, compare ECDFs, Q-Q relationships, percentile bias, and upper-tail frequencies rather than relying only on mean bias or RMSE.

In [ ]:
def percentile_summary(data, percentiles=(1,5,10,50,90,95,99)):
    return data.quantile(np.asarray(percentiles)/100, dim='time')

def percentile_bias(reference, simulation, percentiles=(1,5,10,50,90,95,99)):
    q = np.asarray(percentiles)/100
    return simulation.quantile(q,dim='time') - reference.quantile(q,dim='time')

## 23. Upper-tail diagnostics

In [ ]:
def upper_tail_frequency(data, threshold): return (data > threshold).mean('time')

## 24. Climate-change signal preservation

Compare raw-model and corrected-model future-minus-historical changes. A correction that strongly modifies the projected signal requires careful investigation.

In [ ]:
def period_change(data, hist_start, hist_end, fut_start, fut_end):
    h = data.sel(time=slice(hist_start,hist_end)).mean('time')
    f = data.sel(time=slice(fut_start,fut_end)).mean('time')
    return f-h

## 25. Mean versus median ensemble

In [ ]:
def compare_mean_median(data): return data.mean('model'), data.median('model')

## 26. P10–P90 spread

Percentile ranges can be more stable than min–max for small ensembles. They remain a summary of model outcomes, not automatically probability bounds.

In [ ]:
def p10_p90_spread(data):
    q = data.quantile([.10,.90], dim='model')
    return q.sel(quantile=.90) - q.sel(quantile=.10)

## 27. Publication-quality ensemble time series

In [ ]:
def plot_ensemble_timeseries(ensemble_mean, ensemble_min=None, ensemble_max=None, title='Multi-model ensemble', ylabel=None):
    plt.figure(figsize=(12,5))
    dims=[d for d in ensemble_mean.dims if d!='time']
    y=ensemble_mean.mean(dims,skipna=True)
    plt.plot(y.time,y,label='Ensemble mean')
    if ensemble_min is not None and ensemble_max is not None:
        lo=ensemble_min.mean([d for d in ensemble_min.dims if d!='time'],skipna=True)
        hi=ensemble_max.mean([d for d in ensemble_max.dims if d!='time'],skipna=True)
        plt.fill_between(y.time.values,lo.values,hi.values,alpha=.2,label='Model range')
    plt.xlabel('Time'); plt.ylabel(ylabel or VARIABLE); plt.title(title); plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 28. Trend distribution

In [ ]:
def plot_trend_distribution(trend_values, title='Distribution of model trends'):
    vals=np.asarray(trend_values,float); vals=vals[np.isfinite(vals)]
    plt.figure(figsize=(8,5)); plt.hist(vals,bins=min(10,max(3,len(vals)))); plt.axvline(0,linestyle='--')
    plt.xlabel('Trend'); plt.ylabel('Number of models'); plt.title(title); plt.tight_layout(); plt.show()

## 29. Spatial agreement and uncertainty maps

In [ ]:
def plot_spatial_agreement(agreement, title='Model agreement'):
    plt.figure(figsize=(9,5)); agreement.plot(); plt.title(title); plt.tight_layout(); plt.show()

def plot_uncertainty(spread, title='Model uncertainty / spread'):
    plt.figure(figsize=(9,5)); spread.plot(); plt.title(title); plt.tight_layout(); plt.show()

## 30. Save weights and ensemble products

In [ ]:
def save_weights(weights,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); weights.rename('weight').to_csv(path); print('Saved:',path)

def save_dataset(ds,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); ds.to_netcdf(path); print('Saved:',path)

## 31. Ensemble metadata

In [ ]:
def add_ensemble_metadata(ds, variable, scenario, method, models):
    ds=ds.copy(); ds.attrs.update({'variable':variable,'scenario':scenario,'bias_correction_method':method,'calibration_period':f'{HIST_START} to {HIST_END}','number_of_models':len(models),'model_members':', '.join(map(str,models)),'processing_note':'Multi-model ensemble statistics after model-level bias correction.'}); return ds

## 32. Final ensemble dataset

In [ ]:
def build_final_ensemble_dataset(model_data):
    return xr.Dataset({'ensemble_mean':model_data.mean('model'),'ensemble_median':model_data.median('model'),'model_std':model_data.std('model'),'model_min':model_data.min('model'),'model_max':model_data.max('model'),'p10':model_data.quantile(.10,dim='model'),'p25':model_data.quantile(.25,dim='model'),'p75':model_data.quantile(.75,dim='model'),'p90':model_data.quantile(.90,dim='model')})

## 33. QC: model count

In [ ]:
def qc_model_count(data, expected=None):
    n=int(data.sizes.get('model',0)); print('Models in ensemble:',n)
    if expected is not None and n != expected: print(f'WARNING: expected {expected} models but found {n}')

## 34. QC: missing values

In [ ]:
def qc_missing_fraction(data): return float(data.isnull().mean())

## 35. QC: physical plausibility

For temperature in °C, broad limits can flag corruption. For precipitation, negative values should be investigated. Thresholds are diagnostics, not universal physical limits.

In [ ]:
def temperature_qc(data, lower=-90, upper=70):
    bad=(data<lower)|(data>upper); print(f'Outside [{lower},{upper}] °C: {float(bad.mean()):.6%}'); return bad

def precipitation_qc(data):
    bad=data<0; print(f'Negative fraction: {float(bad.mean()):.6%}'); return bad

## 36. Uncertainty reporting

Report central tendency together with model spread. Do not describe model P10–P90 as formal probability intervals unless a probabilistic framework supports that interpretation.

In [ ]:
def uncertainty_summary(values,label):
    v=np.asarray(values,float); v=v[np.isfinite(v)]
    return pd.Series({'indicator':label,'model_count':len(v),'mean':np.mean(v) if len(v) else np.nan,'median':np.median(v) if len(v) else np.nan,'std':np.std(v,ddof=1) if len(v)>1 else np.nan,'min':np.min(v) if len(v) else np.nan,'max':np.max(v) if len(v) else np.nan,'p10':np.quantile(v,.10) if len(v) else np.nan,'p90':np.quantile(v,.90) if len(v) else np.nan})

## 37. Main analysis versus sensitivity analysis

**Main analysis:** independently corrected models, equal-weight ensemble, spread/P10–P90, agreement, and historical–future change.

**Sensitivity analysis:** median ensemble, skill weighting, alternative correction methods, alternative calibration periods, and alternative model subsets.

## 38. Recommended interpretation

Use statements such as “the ensemble mean indicates…” rather than implying certainty. Model spread represents disagreement among the available models and should be presented as uncertainty information.

## 39. Final production sequence

Raw model data → coordinate/metadata harmonization → independent calibration/validation split → bias correction → validation → corrected model archive → equal-weight ensemble → optional skill-weighted ensemble → spread/percentiles → seasonal and annual diagnostics → extremes → trends → agreement → uncertainty reporting → NetCDF/CSV/figures.

## 40. Final research checklist

- [ ] Expected models included
- [ ] No duplicate models
- [ ] Coordinates compatible
- [ ] Time coordinates valid
- [ ] Calibration and validation periods independent
- [ ] Future data not used to fit correction
- [ ] Units consistent
- [ ] Physical QC completed
- [ ] Missing values quantified
- [ ] Extreme-event behavior checked
- [ ] Climate-change signal preservation checked
- [ ] Ensemble spread reported
- [ ] Model agreement reported
- [ ] Weighting assumptions documented
- [ ] NetCDF/CSV metadata complete
- [ ] Intermediate corrected model files retained

# Conclusion

Part 5 completes the uncertainty and production layer of Notebook 06. The preferred baseline is to correct each model independently, validate using an independent period, retain all corrected members, construct an equal-weight ensemble, report model spread and agreement, and use weighting or alternative correction methods as sensitivity analyses.